In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]

docs = WebBaseLoader(urls).load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splitted_docs = text_splitter.split_documents(docs)

embd = OpenAIEmbeddings()

vector_store = FAISS.from_documents(
    splitted_docs,
    embd 
)

retriever = vector_store.as_retriever()


In [36]:
# Router
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class RouterQuery(BaseModel):
    """Route a user query to most relevant datasource."""
    datasource: Literal["vectorstore", "websearch"] = Field(description="Given a user question choose to route it websearch or vectorstore.")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_router = llm.with_structured_output(RouterQuery)

# prompt
system = """
You are an expert in routing a user question to a vectorstore or websearch.
The vectorstore conatins documents realted to agent, prompt engineering, and adversarial attack.
Use the vectorstore for questionson these topics. Otherwise, use websearch
"""
route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}")
    ]
)

question_router = route_prompt | structured_llm_router


In [38]:
question_router.invoke({"question": "What are the types of agent memory?"})

c:\Codebase\GenAI\langgraph_practice\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouterQuery(datasource='vectorstore'), input_type=RouterQuery])
  return self.__pydantic_serializer__.to_python(


RouterQuery(datasource='vectorstore')

In [ ]:
# Retriever Grader
class GradeDocuments(BaseModel):
    """Binary score on the relevant check on the retrieved document."""
    binary_score: str = Field(description="Documents are relevant to the question 'yes' or 'no'")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

sytem_prompt = """
You are a grader assessing relevance of the retrieved documents to a user question.
If the document conatins keyword(s) or semantic meaning related to user question, grade it as relevant.
It does not need to be a stringent test. The goal is filter out errorneous retrievals.
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.
"""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", sytem_prompt),
        ("human", "Retrieved documents: {documents}\n\n User question: {question}\n\n")
    ]
)